# # Inference: Processing Massive Log Files with Generators in Python

A runnable Google Colab notebook to process large log files with minimal memory using generators, and calculate unique daily users.

**Run this notebook top to bottom.**

This notebook demonstrates how to process large log files efficiently using Python generators to keep memory usage minimal. We'll generate a synthetic log file and then process it to count unique daily users.

In [ ]:
import time
import random
from datetime import datetime, timedelta
import os

# Define the filename for our synthetic log
LOG_FILENAME = 'synthetic.log'

# This function generates a synthetic log file. It writes lines directly to disk
# without holding the entire file in memory, mimicking a real log stream.
print(f"Generating synthetic log file: {LOG_FILENAME}...")
start_gen_time = time.time()

# Let's create a log file with 1 million lines to keep the runtime under a minute.
# The original brief mentioned 10 million, but that would exceed the 15-second cell limit.
# The principle remains the same.
num_lines = 1_000_000
start_date = datetime(2023, 1, 1)

with open(LOG_FILENAME, 'w') as f:
    for i in range(num_lines):
        # Generate a random date within a year
        random_days = random.randint(0, 364)
        current_date = start_date + timedelta(days=random_days)
        timestamp = current_date.strftime('%Y-%m-%d %H:%M:%S')

        # Generate a random user ID
        user_id = f'user_{random.randint(1, 10000)}'

        # Write the log line to the file
        f.write(f"{timestamp},{user_id}\n")

end_gen_time = time.time()
print(f"Generated {num_lines} lines in {end_gen_time - start_gen_time:.2f} seconds.")
print(f"Log file '{LOG_FILENAME}' created.")

Now, we define a generator function to read the log file line by line. This is the core of our memory-efficient processing. It yields one line at a time, ensuring we never load the whole file into RAM.

In [ ]:
# This generator function reads the log file line by line.
# It yields each line without storing the entire file in memory.
def read_log_with_generator(filename):
    print(f"Starting to read log file '{filename}' using a generator...")
    with open(filename, 'r') as f:
        for line in f:
            yield line.strip() # .strip() removes leading/trailing whitespace, including newline characters

# We can test our generator by printing the first few lines.
print("Testing the generator with the first 5 lines:")
log_generator = read_log_with_generator(LOG_FILENAME)
for i in range(5):
    try:
        print(next(log_generator))
    except StopIteration:
        print("End of file reached before 5 lines.")
        break

With our generator in place, we can now process the log file to count unique daily users. We'll store unique user IDs for each day. The key here is that we're not building a massive list of all users for all days at once. We process each line and update our counts incrementally. This is where the 'honest limitation' of potentially slow processing comes in for large files, as we iterate through every single line.

In [ ]:
from collections import defaultdict

# This dictionary will store unique users for each day.
# `defaultdict(set)` is perfect because it automatically creates a new set
# for a date if it's encountered for the first time.
unique_daily_users = defaultdict(set)

print("Processing log file to count unique daily users...")
start_process_time = time.time()

# Iterate through the log file using our generator
log_generator = read_log_with_generator(LOG_FILENAME)

processed_lines_count = 0
for line in log_generator:
    try:
        timestamp_str, user_id = line.split(',', 1)
        # Extract the date part from the timestamp
        log_date = timestamp_str.split(' ')[0]

        # Add the user ID to the set for the corresponding day
        unique_daily_users[log_date].add(user_id)
        processed_lines_count += 1

        # Print progress every 100,000 lines to show it's working
        if processed_lines_count % 100_000 == 0:
            print(f"Processed {processed_lines_count} lines...")

    except ValueError:
        # Handle lines that might not be in the expected format (e.g., missing comma)
        print(f"Skipping malformed line: {line}")
    except Exception as e:
        print(f"An unexpected error occurred processing line '{line}': {e}")


end_process_time = time.time()
processing_duration = end_process_time - start_process_time

print(f"Finished processing {processed_lines_count} lines.")
print(f"Total processing time: {processing_duration:.2f} seconds.")

Now, let's calculate the total number of unique users across all days by summing up the counts for each day. This step is quick because we're just summing already computed set sizes, not re-processing the file.

In [ ]:
# Calculate the total number of unique users across all days
total_unique_users = sum(len(users) for users in unique_daily_users.values())

print(f"Total number of unique users across all days: {total_unique_users}")

# Display the unique user count for a few sample days
print("\nSample daily unique user counts:")
sample_dates = sorted(unique_daily_users.keys())[:5]
for date in sample_dates:
    print(f"- {date}: {len(unique_daily_users[date])} unique users")

Finally, we'll save the calculated unique daily user counts to a CSV file. This artifact can be used for further analysis or reporting. The notebook also highlights the honest limitation: while memory is managed, processing millions of lines sequentially can still take a significant amount of time.

In [ ]:
import pandas as pd

# Convert the defaultdict to a pandas DataFrame for easy saving to CSV
# We create a list of dictionaries, where each dictionary represents a row
data_for_csv = []
for date, users in unique_daily_users.items():
    data_for_csv.append({'date': date, 'unique_user_count': len(users)})

df_unique_users = pd.DataFrame(data_for_csv)

# Sort by date for a clean CSV output
df_unique_users = df_unique_users.sort_values(by='date').reset_index(drop=True)

# Define the output CSV filename
OUTPUT_CSV_FILENAME = 'unique_daily_users.csv'

# Save the DataFrame to a CSV file
df_unique_users.to_csv(OUTPUT_CSV_FILENAME, index=False)

print(f"\nUnique daily user counts saved to '{OUTPUT_CSV_FILENAME}'.")
print(f"You can find this file in the Colab file browser (left sidebar).")

print("\nThis notebook successfully demonstrated processing large log files with generators, keeping memory usage minimal.")
print("The honest limitation is that while memory is managed, the time taken to process millions of lines sequentially can still be substantial.")